# TerraSeg AI - Phase 3: Better Loss, Attention, Augmentation
**Notebook 03 - Google Colab, GPU (T4). Fully self-contained: run top to bottom in a fresh runtime.**

### Where we start
Phase 2A (Landslide4Sense) gave a clear winner: **ResNet34 U-Net on 14 channels (12 Sentinel-2 bands + slope + DEM)**, IoU 0.621. That model is now our **baseline `E0`**. Phase 3 tries to beat it with the improvements from your proposal.

We stay on Landslide4Sense because that pipeline works and its results are meaningful. (The Sen12Landslides run in Phase 2B did not give usable numbers yet, see the notes in chat.)

### The experiments (one change at a time, then all together)
| ID | Loss | Decoder attention | Augmentation | Question |
|---|---|---|---|---|
| E0 | BCE + Dice | none | flips | baseline (re-run under the new, cleaner protocol) |
| E1 | **Focal + Dice** | none | flips | does Focal loss help with rare landslide pixels? |
| E2 | BCE + Dice | **scSE** | flips | does attention sharpen boundaries? |
| E3 | BCE + Dice | none | **flips + 90 deg rotations** | does more augmentation help? |
| E4 | Focal + Dice | scSE | flips + rotations | everything together |

### What is new in the protocol (more honest than Phase 2A)
1. **Train / validation / test = 70 / 15 / 15.** Validation picks the best epoch; the **test set is only used for final numbers**. In Phase 2A the same 20% did both jobs, which is slightly optimistic.
2. **Checkpoint chosen by validation IoU** (the metric we care about), not by loss, so different loss functions are comparable.
3. **The "best" experiment is chosen using validation scores**, never test scores.
4. **Several seeds** for the important runs, so we know what is real and what is noise.
5. Extras that cost no training: **threshold tuning**, **flip test-time augmentation**, and a **boundary F-score** (your project is about boundary mapping, so we measure boundaries, not just overlap).

### About "attention"
scSE (concurrent spatial and channel squeeze-and-excitation) is built into `segmentation_models_pytorch`. It lets the decoder re-weight feature channels and pixels, so it focuses on informative regions. It is a lighter stand-in for the C2PSA module in your proposal; describe it as "scSE attention" in the report.

## Part 1 - Setup and data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install segmentation_models_pytorch h5py tqdm

In [ ]:
import os, glob, re, random, time, json, shutil
import numpy as np, pandas as pd, h5py, torch
import torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED_SPLIT = 42                      # fixed split, never changes
OUT = '/content/drive/MyDrive/TerraSeg/phase3'
os.makedirs(f'{OUT}/runs', exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device, "| outputs ->", OUT)
if device.type == 'cpu':
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU")

### 1.1 Download Landslide4Sense (train part, ~2.5 GB, fetched directly inside Colab) and load it
Same loading code that worked in Phase 2A: images from `images/train`, masks from `annotations/train` (the stray files in the mask folder are ignored by the folder-specific patterns).

A few patches in this dataset can be entirely empty (all zeros). We find them and exclude them from training and testing **without copying the big array**.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="ibm-nasa-geospatial/Landslide4sense", repo_type="dataset",
                  allow_patterns=["*train*"], local_dir="/content/l4s", max_workers=8)

def num(p): return int(re.findall(r'(\d+)\.h5$', p)[0])
img_files  = sorted(glob.glob('/content/l4s/images/train/image_*.h5'), key=num)
mask_files = sorted(glob.glob('/content/l4s/annotations/train/mask_*.h5'), key=num)
print(len(img_files), "images |", len(mask_files), "masks")
assert len(img_files) == len(mask_files) > 0
assert all(num(a) == num(b) for a, b in zip(img_files, mask_files)), "image/mask mismatch"

def first(h): return h[list(h.keys())[0]][:]
N = len(img_files)
X = np.zeros((N, 14, 128, 128), dtype=np.float32)
Y = np.zeros((N, 1, 128, 128), dtype=np.float32)
for i, (a, b) in enumerate(tqdm(list(zip(img_files, mask_files)))):
    with h5py.File(a) as h: X[i] = np.nan_to_num(first(h)).transpose(2, 0, 1)
    with h5py.File(b) as h: Y[i, 0] = first(h)

valid = np.array([np.abs(X[i, :12]).max() > 0 for i in range(N)])
valid_idx = np.where(valid)[0]
print(f"patches {N} | empty (all-zero) patches excluded: {(~valid).sum()} | usable: {len(valid_idx)}")
print(f"landslide pixel share (usable patches): {Y[valid_idx].mean()*100:.2f}%")

### 1.2 Fixed split + per-channel normalisation
- 70 / 15 / 15 random patch split, seed fixed and saved to Drive.
- **Honest limitation:** this dataset has no region labels in the file names, so the split is random per patch. Neighbouring patches can end up in different sets, so absolute scores are somewhat optimistic. The comparison **between experiments** is still fair because all share the split.
- Mean and std come from **training patches only**, then the whole array is normalised **in place** (saves about 3.5 GB of RAM compared to making a copy).

In [ ]:
rng = np.random.RandomState(SEED_SPLIT)
perm = rng.permutation(valid_idx)
n = len(perm); n_tr, n_va = int(0.70 * n), int(0.15 * n)
train_idx, val_idx, test_idx = perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]
np.savez(f'{OUT}/split.npz', train=train_idx, val=val_idx, test=test_idx)
print(f"train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")

if globals().get('NORMALISED', False):
    raise RuntimeError("X is already normalised. Running this cell twice corrupts the saved statistics. "
                       "Use Runtime -> Restart session and run from the top.")

s1 = np.zeros(14, np.float64); s2 = np.zeros(14, np.float64); cnt = 0
for s in range(0, len(train_idx), 256):
    b = X[np.sort(train_idx[s:s + 256])].astype(np.float64)
    s1 += b.sum(axis=(0, 2, 3)); s2 += (b ** 2).sum(axis=(0, 2, 3)); cnt += b.shape[0] * 128 * 128
MEAN = (s1 / cnt).astype(np.float32)
STD = (np.sqrt(np.maximum(s2 / cnt - (s1 / cnt) ** 2, 1e-12)) + 1e-6).astype(np.float32)
json.dump({'mean': MEAN.tolist(), 'std': STD.tolist()}, open(f'{OUT}/norm_stats.json', 'w'))

m4, s4 = MEAN.reshape(1, 14, 1, 1), STD.reshape(1, 14, 1, 1)
for s in range(0, N, 256):
    X[s:s + 256] = (X[s:s + 256] - m4) / s4
NORMALISED = True
print("normalised. channel 13 (DEM) raw mean/std:", round(float(MEAN[13]), 1), round(float(STD[13]), 1))
print(f"landslide share in TRAIN {Y[train_idx].mean()*100:.2f}% | VAL {Y[val_idx].mean()*100:.2f}% | TEST {Y[test_idx].mean()*100:.2f}%")

## Part 2 - Dataset, losses, model, metrics

### 2.1 Dataset with two augmentation levels
- `flip`: random horizontal and vertical flips (what Phase 2A used).
- `flip_rot`: flips **plus** random 90-degree rotations. Satellite images have no "right way up", so rotations are label-preserving. Applied identically to all 14 channels and the mask.
- Validation and test are never augmented.

In [ ]:
ALL = list(range(14))

class PatchDS(Dataset):
    def __init__(self, idx, aug='none'):
        self.idx, self.aug = np.asarray(idx), aug
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = torch.from_numpy(X[j].copy())
        y = torch.from_numpy(Y[j].copy())
        if self.aug in ('flip', 'flip_rot'):
            if random.random() > 0.5: x, y = x.flip(-1), y.flip(-1)
            if random.random() > 0.5: x, y = x.flip(-2), y.flip(-2)
        if self.aug == 'flip_rot':
            k = random.randint(0, 3)
            if k: x, y = torch.rot90(x, k, (1, 2)), torch.rot90(y, k, (1, 2))
        return x, y

def loader(idx, aug='none', shuffle=False, bs=16, seed=0):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(PatchDS(idx, aug), batch_size=bs, shuffle=shuffle, generator=g if shuffle else None,
                      num_workers=2, pin_memory=True)

print("Dataset ready")

### 2.2 Losses
**BCE + Dice** (your current loss): BCE gives stable per-pixel learning, Dice pushes for overlap.

**Focal + Dice** (new): Focal loss multiplies the BCE of every pixel by `(1 - p_correct)^gamma`. Easy pixels (the huge background areas the model already gets right) contribute almost nothing, so the training signal is dominated by the hard pixels: landslide pixels and confusing background. `alpha=0.75` gives the rare landslide class more weight. We divide by the number of landslide pixels (the standard RetinaNet normalisation) so Focal and Dice have a similar size.

In [ ]:
def dice_term(logits, t):
    p, tf = torch.sigmoid(logits).view(-1), t.view(-1)
    return 1 - (2 * (p * tf).sum() + 1e-6) / (p.sum() + tf.sum() + 1e-6)

class BCEDiceLoss(nn.Module):
    def forward(self, logits, t):
        return F.binary_cross_entropy_with_logits(logits, t) + dice_term(logits, t)

class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__(); self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, t):
        bce = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
        p = torch.sigmoid(logits)
        pt = p * t + (1 - p) * (1 - t)                       # probability given to the correct class
        a = self.alpha * t + (1 - self.alpha) * (1 - t)
        focal = (a * (1 - pt) ** self.gamma * bce).sum() / t.sum().clamp(min=1.0)
        return focal + dice_term(logits, t)

LOSSES = {'bce_dice': BCEDiceLoss, 'focal_dice': FocalDiceLoss}

def build(attn=None, pretrained=True):
    return smp.Unet('resnet34', encoder_weights='imagenet' if pretrained else None,
                    in_channels=14, classes=1, decoder_attention_type=attn).to(device)

print("Losses and model builder ready")

### 2.3 Metrics, prediction with optional test-time augmentation
- Counts (TP, FP, FN, TN) are summed over the **whole** set, as in Phase 1.
- **TTA (test-time augmentation):** predict on the image and on its flipped versions, flip the predictions back, and average. No retraining, usually a small free gain.

In [ ]:
def metrics_from_counts(tp, fp, fn, tn):
    e = 1e-9
    prec, rec = tp / (tp + fp + e), tp / (tp + fn + e)
    return {'IoU': tp / (tp + fp + fn + e), 'F1': 2 * prec * rec / (prec + rec + e),
            'Precision': prec, 'Recall': rec, 'PixelAcc': (tp + tn) / (tp + tn + fp + fn + e)}

@torch.no_grad()
def predict_probs(model, x, tta=False):
    model.eval()
    p = torch.sigmoid(model(x))
    if not tta:
        return p
    p = p + torch.sigmoid(model(x.flip(-1))).flip(-1)
    p = p + torch.sigmoid(model(x.flip(-2))).flip(-2)
    p = p + torch.sigmoid(model(x.flip([-1, -2]))).flip([-1, -2])
    return p / 4

@torch.no_grad()
def collect_probs(model, ldr, tta=False):
    P, T = [], []
    for x, y in ldr:
        P.append(predict_probs(model, x.to(device), tta)[:, 0].cpu().numpy().astype(np.float16))
        T.append(y[:, 0].numpy().astype(np.uint8))
    return np.concatenate(P), np.concatenate(T)

def metrics_from_probs(P, T, thr=0.5):
    p, t = P.astype(np.float32) > thr, T == 1
    return metrics_from_counts(float((p & t).sum()), float((p & ~t).sum()), float((~p & t).sum()), float((~p & ~t).sum()))

def evaluate(model, ldr, thr=0.5, tta=False):
    P, T = collect_probs(model, ldr, tta)
    return metrics_from_probs(P, T, thr)

print("Metrics ready")

## Part 3 - Training the experiments

### 3.1 Experiment definitions and the (resumable) training function
- Same everything except the one change per experiment: Adam, lr 5e-4, batch 16, up to 40 epochs, ImageNet-pretrained ResNet-34 encoder.
- The best epoch is chosen by **validation IoU**. Early stopping after 10 epochs without improvement.
- Every finished run is saved to Drive (`runs/<exp>_s<seed>.json` + weights). If Colab disconnects, re-run the notebook and finished runs are skipped.
- Results store **both validation and test** scores. Validation is used to choose; test is only for reporting.

In [ ]:
EXPERIMENTS = {
    'E0_baseline': dict(loss='bce_dice',   attn=None,   aug='flip'),
    'E1_focal':    dict(loss='focal_dice', attn=None,   aug='flip'),
    'E2_scse':     dict(loss='bce_dice',   attn='scse', aug='flip'),
    'E3_rotaug':   dict(loss='bce_dice',   attn=None,   aug='flip_rot'),
    'E4_combined': dict(loss='focal_dice', attn='scse', aug='flip_rot'),
}
EPOCHS, PATIENCE, LR, FORCE = 40, 10, 5e-4, False

def rpath(name, seed): return f'{OUT}/runs/{name}_s{seed}.json'
def cpath(name, seed): return f'{OUT}/runs/{name}_s{seed}.pth'

def run_experiment(name, seed):
    if os.path.exists(rpath(name, seed)) and not FORCE:
        print(f"skip {name} seed {seed} (done)")
        return json.load(open(rpath(name, seed)))
    cfg = EXPERIMENTS[name]
    print("\n" + "=" * 72 + f"\n{name} | seed {seed} | {cfg}\n" + "=" * 72)
    set_seed(seed)
    tr = loader(train_idx, cfg['aug'], True, 16, seed)
    va, te = loader(val_idx, 'none', False, 32), loader(test_idx, 'none', False, 32)
    model = build(cfg['attn']); crit = LOSSES[cfg['loss']]()
    opt = optim.Adam(model.parameters(), lr=LR)
    best, bad, ran = -1.0, 0, 0
    for ep in range(EPOCHS):
        model.train(); run = 0.0
        for x, y in tr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step(); run += loss.item()
        v = evaluate(model, va); ran = ep + 1
        print(f"[{name}/{seed}] ep {ran:02d} train {run/len(tr):.4f} | val IoU {v['IoU']:.4f} F1 {v['F1']:.4f}")
        if v['IoU'] > best:
            best, bad = v['IoU'], 0; torch.save(model.state_dict(), cpath(name, seed))
        else:
            bad += 1
            if bad >= PATIENCE:
                print("early stopping"); break
    model.load_state_dict(torch.load(cpath(name, seed), map_location=device))
    res = {'name': name, 'seed': seed, 'cfg': cfg, 'epochs_run': ran,
           'val': evaluate(model, va), 'test': evaluate(model, te)}
    json.dump(res, open(rpath(name, seed), 'w'))
    print(f"TEST -> IoU {res['test']['IoU']:.4f} | F1 {res['test']['F1']:.4f} | "
          f"P {res['test']['Precision']:.4f} | R {res['test']['Recall']:.4f}")
    del model, opt; torch.cuda.empty_cache()
    return res

print("Ready. Experiments:", list(EXPERIMENTS))

### 3.2 Screening run: one seed for every experiment
This is the quick pass (5 runs). It tells us which ideas look promising.

In [ ]:
for name in EXPERIMENTS:
    run_experiment(name, 42)
print("\nScreening finished.")

### 3.3 Results table
Read it like this:
- `IoU gain vs E0` = test IoU minus the baseline's test IoU.
- With one seed, gaps of about 0.01 to 0.02 can be noise. Do not claim anything from them yet.
- The **best experiment is picked by mean validation IoU** (`val_IoU`), not by the test column.

In [ ]:
def load_all_runs():
    rows = []
    for f in sorted(glob.glob(f'{OUT}/runs/*.json')):
        r = json.load(open(f))
        row = {'exp': r['name'], 'seed': r['seed']}
        row.update({f'test_{k}': v for k, v in r['test'].items()})
        row.update({f'val_{k}': v for k, v in r['val'].items()})
        rows.append(row)
    return pd.DataFrame(rows)

def summarise():
    runs = load_all_runs()
    g = runs.groupby('exp')
    tab = pd.DataFrame({
        'runs': g.size(),
        'IoU': g['test_IoU'].mean(), 'IoU std': g['test_IoU'].std(),
        'F1': g['test_F1'].mean(), 'Precision': g['test_Precision'].mean(), 'Recall': g['test_Recall'].mean(),
        'val_IoU': g['val_IoU'].mean(),
    }).reindex([e for e in EXPERIMENTS if e in g.groups])
    tab['IoU gain vs E0'] = tab['IoU'] - tab.loc['E0_baseline', 'IoU']
    return tab

tab = summarise()
display(tab.round(4))
tab.round(4).to_csv(f'{OUT}/phase3_results.csv')
best_exp = tab['val_IoU'].idxmax()
print("Best experiment by VALIDATION IoU:", best_exp)

xs = np.arange(len(tab)); w = 0.38
err = tab['IoU std'].fillna(0).values if tab['runs'].max() > 1 else None
plt.figure(figsize=(9, 4))
plt.bar(xs - w / 2, tab['IoU'].values, w, yerr=err, capsize=3, label='IoU')
plt.bar(xs + w / 2, tab['F1'].values, w, label='F1')
plt.xticks(xs, tab.index, rotation=20); plt.legend()
plt.title("Phase 3: test IoU and F1 per experiment (error bar = std over seeds)")
plt.tight_layout(); plt.savefig(f'{OUT}/phase3_results.png', dpi=150); plt.show()

### 3.4 Confirm with more seeds (do this before believing any gap)
Re-runs three configurations with seeds 123 and 777: the baseline, the best screening experiment, and the full combination `E4`. Takes roughly 4 to 6 more runs. Finished runs are skipped, so it is safe to re-run.

In [ ]:
to_confirm = list(dict.fromkeys(['E0_baseline', best_exp, 'E4_combined']))
print("Confirming:", to_confirm)
for seed in (123, 777):
    for name in to_confirm:
        run_experiment(name, seed)

tab = summarise()
display(tab.round(4))
tab.round(4).to_csv(f'{OUT}/phase3_results.csv')
final_exp = tab[tab['runs'] >= 3]['val_IoU'].idxmax()
print("\nFINAL experiment (best mean validation IoU among 3-seed configs):", final_exp)

## Part 4 - Free improvements on the final model: threshold and TTA

The model outputs a probability per pixel and we call it landslide above 0.5. But 0.5 is arbitrary, especially with rare positives.
1. **Pick the threshold on the validation set** (try 0.10 to 0.90, keep the best IoU).
2. Apply that threshold to the **test set** (never tune on test).
3. Do it with and without **flip TTA**.

In [ ]:
def load_model(name, seed):
    m = build(EXPERIMENTS[name]['attn'], pretrained=False)
    m.load_state_dict(torch.load(cpath(name, seed), map_location=device))
    return m.eval()

final_model = load_model(final_exp, 42)
va_l, te_l = loader(val_idx, 'none', False, 32), loader(test_idx, 'none', False, 32)

rows, chosen = [], {}
for tta in (False, True):
    Pv, Tv = collect_probs(final_model, va_l, tta)
    Pt, Tt = collect_probs(final_model, te_l, tta)
    thrs = np.arange(0.10, 0.91, 0.05)
    val_iou = [metrics_from_probs(Pv, Tv, t)['IoU'] for t in thrs]
    best_thr = float(thrs[int(np.argmax(val_iou))])
    chosen[tta] = best_thr
    for label, thr in (('default 0.50', 0.5), (f'tuned {best_thr:.2f}', best_thr)):
        m = metrics_from_probs(Pt, Tt, thr)
        rows.append({'TTA': tta, 'threshold': label, **{k: round(m[k], 4) for k in ('IoU', 'F1', 'Precision', 'Recall')}})
free = pd.DataFrame(rows)
display(free)
free.to_csv(f'{OUT}/phase3_threshold_tta.csv', index=False)

## Part 5 - Boundaries (the "mapping" in landslide detection and boundary mapping)

IoU rewards getting the area right; it says less about how well the **outline** is drawn. **Boundary F-score** checks the outline directly:
- Extract the boundary pixels of the predicted and true masks.
- *Boundary precision*: share of predicted boundary pixels within 2 pixels (about 20 m) of a true boundary.
- *Boundary recall*: share of true boundary pixels within 2 pixels of a predicted boundary.
- F-score = balance of the two. Averaged over test patches that contain a landslide.

We compare the **baseline E0** with the **final model** (same threshold rule: tuned on validation).

In [ ]:
from scipy.ndimage import binary_erosion, distance_transform_edt

def boundary(mask):
    return mask & ~binary_erosion(mask, border_value=0)

def boundary_f1(pred, gt, tol=2):
    bp, bg = boundary(pred), boundary(gt)
    if bg.sum() == 0 and bp.sum() == 0:
        return None                                   # nothing to compare
    if bg.sum() == 0 or bp.sum() == 0:
        return 0.0
    d_to_gt, d_to_pred = distance_transform_edt(~bg), distance_transform_edt(~bp)
    prec, rec = (d_to_gt[bp] <= tol).mean(), (d_to_pred[bg] <= tol).mean()
    return 0.0 if prec + rec == 0 else float(2 * prec * rec / (prec + rec))

def mean_boundary_f1(P, T, thr, tol=2):
    vals = [boundary_f1(P[i].astype(np.float32) > thr, T[i] == 1, tol) for i in range(len(P)) if T[i].any()]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)), len(vals)

base_model = load_model('E0_baseline', 42)
Pv_b, Tv_b = collect_probs(base_model, va_l, False)
Pt_b, Tt_b = collect_probs(base_model, te_l, False)
thrs = np.arange(0.10, 0.91, 0.05)
thr_b = float(thrs[int(np.argmax([metrics_from_probs(Pv_b, Tv_b, t)['IoU'] for t in thrs]))])

Pt_f, Tt_f = collect_probs(final_model, te_l, True)
thr_f = chosen[True]

rows = []
for label, P, T, thr in (('E0 baseline (tuned thr)', Pt_b, Tt_b, thr_b), (f'{final_exp} (tuned thr + TTA)', Pt_f, Tt_f, thr_f)):
    m = metrics_from_probs(P, T, thr)
    bf, n_used = mean_boundary_f1(P, T, thr)
    rows.append({'model': label, 'threshold': round(thr, 2), 'IoU': round(m['IoU'], 4), 'F1': round(m['F1'], 4),
                 'Boundary F1 (2px)': round(bf, 4), 'patches': n_used})
bt = pd.DataFrame(rows); display(bt)
bt.to_csv(f'{OUT}/phase3_boundary.csv', index=False)

In [ ]:
# error maps on the same test patches:  green = correct landslide, red = false alarm, blue = missed landslide
sel = [i for i in range(len(Tt_f)) if Tt_f[i].sum() >= 300][:4]
def err_map(pred, gt):
    img = np.zeros(pred.shape + (3,), np.float32)
    img[pred & gt] = (0, 1, 0); img[pred & ~gt] = (1, 0, 0); img[~pred & gt] = (0, 0.4, 1)
    return img

fig, ax = plt.subplots(len(sel), 4, figsize=(13, 3.2 * len(sel)))
for r, i in enumerate(sel):
    j = test_idx[i]
    rgb = (X[j][[3, 2, 1]] * STD[[3, 2, 1]].reshape(3, 1, 1) + MEAN[[3, 2, 1]].reshape(3, 1, 1)).transpose(1, 2, 0)
    lo, hi = np.percentile(rgb, 2), np.percentile(rgb, 98)
    ax[r, 0].imshow(np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)); ax[r, 0].set_title("Optical RGB")
    ax[r, 1].imshow(Tt_f[i], cmap='gray'); ax[r, 1].set_title("Ground truth")
    ax[r, 2].imshow(err_map(Pt_b[i].astype(np.float32) > thr_b, Tt_b[i] == 1)); ax[r, 2].set_title("E0 baseline")
    ax[r, 3].imshow(err_map(Pt_f[i].astype(np.float32) > thr_f, Tt_f[i] == 1)); ax[r, 3].set_title(f"{final_exp}")
    for a in ax[r]: a.axis('off')
plt.suptitle("green = correct | red = false alarm | blue = missed", y=1.0)
plt.tight_layout(); plt.savefig(f'{OUT}/phase3_error_maps.png', dpi=150); plt.show()

## Part 6 - Save the final model (needed for Phase 5: Grad-CAM and the web demo)

In [ ]:
shutil.copy(cpath(final_exp, 42), f'{OUT}/final_model.pth')
final_cfg = {'experiment': final_exp, 'config': EXPERIMENTS[final_exp], 'seed': 42,
             'threshold': chosen[True], 'tta': True, 'in_channels': 14,
             'mean': MEAN.tolist(), 'std': STD.tolist(),
             'channel_order': [f'S2_B{i+1}' for i in range(12)] + ['slope', 'DEM']}
json.dump(final_cfg, open(f'{OUT}/final_config.json', 'w'), indent=2)
print("Saved final_model.pth and final_config.json in", OUT)
print(json.dumps({k: v for k, v in final_cfg.items() if k not in ('mean', 'std')}, indent=2))

## Phase 3 checklist
- [ ] 5 screening runs finished (seed 42)
- [ ] Baseline, best and E4 confirmed with 3 seeds
- [ ] `phase3_results.csv`, `phase3_threshold_tta.csv`, `phase3_boundary.csv` and the figures saved
- [ ] `final_model.pth` + `final_config.json` saved
- [ ] Notebook pushed to GitHub as `notebooks/03_phase3_improvements.ipynb`

## How to read the outcome
- **Gain larger than the std across seeds:** a real improvement. Report it.
- **No clear gain:** also a finding ("Focal and attention did not help on top of BCE+Dice here"). Do not hide it.
- **Boundary F1 improves even if IoU barely moves:** that supports the proposal's boundary-sharpness claim.

## Next: Phase 5 - Grad-CAM explainability + web demo